# Fine-tuning LoRA de Florence-2 -- clasificador de atributos de prenda

Wrapper fino sobre `entrenar_lora.py` (Stage 3 del plan) para correr el fine-tuning real en
una GPU T4 gratuita de Colab -- en local (CPU, 4GB VRAM sin CUDA funcional) tardaria del
orden de dias en vez de horas. Ver `experimentos/vlm_atributos_prenda/README.md` en el repo
para el contexto completo.

**Antes de correr esto**: activa GPU en `Entorno de ejecucion > Cambiar tipo de entorno de
ejecucion > T4 GPU`.

**Paso 0 (hazlo tu, fuera de este notebook)**: la rama con este codigo tiene que estar
subida a GitHub para poder clonarla aqui (`git push -u origin <rama>` desde tu maquina). Si
el repo es privado, necesitaras un token de acceso personal. Alternativa sin git: comprime a
mano la carpeta `experimentos/vlm_atributos_prenda/` (sin `data/imagenes_cache/`, se
regenera aqui) y subela con el icono de carpeta de la izquierda, y salta la celda de `git
clone` de abajo.

In [ ]:
!nvidia-smi

## 1. Clonar el repo (o saltar si subiste la carpeta a mano)

In [ ]:
RAMA = "clip-trend-semantic-matching-poc"  # ajustar si se ha renombrado/mergeado

!git clone --branch $RAMA --single-branch https://github.com/victor8701/tfm-robotic-picking-vision.git repo
%cd repo/experimentos/vlm_atributos_prenda

## 2. Instalar dependencias

`transformers` fijado a `4.51.3` a proposito -- Florence-2 se rompe con versiones mas nuevas
(ver el aviso en el README del repo).

Esta celda tambien desinstala `torchao`: Colab lo trae preinstalado en una version (0.10.0)
demasiado vieja para la `peft` que instalamos, y `get_peft_model()` falla con
`ImportError: Found an incompatible version of torchao` al intentar aplicar LoRA -- bug
conocido y reportado en varios proyectos, no algo especifico de este notebook. No usamos
cuantizacion en ningun momento, asi que quitarlo es la solucion correcta (no hace falta
actualizarlo).

In [ ]:
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao

## ⚠️ Reinicia la sesion antes de seguir

El `torchao` viejo ya esta cargado en memoria en este proceso de Python aunque lo acabemos
de desinstalar -- si no reinicias, el error de la celda de entrenamiento (4) sigue saliendo
igual. Dos formas de reiniciar:

- Menu **Entorno de ejecucion > Reiniciar sesion** (o el atajo `Ctrl+M .`), o
- Ejecuta la celda de abajo, que lo hace por codigo (Colab avisara con un mensaje de que el
  proceso ha muerto -- es lo esperado, no un fallo).

**Despues de reiniciar, NO vuelvas a correr las celdas 1-3** (ya estan instaladas/clonadas en
el mismo entorno) -- salta directamente a la celda de `%cd repo/experimentos/vlm_atributos_prenda`
de abajo y sigue desde el paso 3 en adelante.

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
# Correr esta celda justo despues de reiniciar la sesion (el cd se pierde al reiniciar)
%cd repo/experimentos/vlm_atributos_prenda

## 3. Regenerar las imagenes de entrenamiento

`data/train.jsonl`, `val.jsonl` y `test.jsonl` ya vienen en el repo (son deterministas, misma
semilla). Las imagenes (`data/imagenes_cache/`) no se suben a git -- se regeneran aqui
descargando de nuevo el parquet de Kaggle/Hugging Face (~270 MB, un par de minutos con la
conexion de Colab) y volviendo a correr el mismo script de preparacion, que es determinista:
reproduce exactamente los mismos `.jsonl` ya commiteados y solo rellena las imagenes que
faltan.

In [ ]:
!python3 herramientas/preparar_dataset_florence2.py

## 4. Entrenar (LoRA, ~3 epocas)

Hiperparametros de partida ya fijados por defecto en `entrenar_lora.py` (ver el script para
el detalle y la justificacion). Con GPU T4, batch 4 + acumulacion 4 (batch efectivo 16) deberia
caber sin problema en las 16GB de VRAM del T4 -- si da error de memoria, bajar `--batch`.

In [ ]:
!python3 herramientas/entrenar_lora.py --salida modelos/florence2_base_lora_v4

## 5. Descargar el adapter entrenado

Son solo unos MB (LoRA, no el modelo entero). Baja el zip y descomprimelo en
`experimentos/vlm_atributos_prenda/modelos/florence2_base_lora_v4/` en tu maquina para poder
correr `evaluar_modelo.py` (Stage 4) en local.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("florence2_base_lora_v4", "zip", "modelos/florence2_base_lora_v4")
files.download("florence2_base_lora_v4.zip")